In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt

train = pd.read_csv("../data/train_data.csv")
test = pd.read_csv("../data/test_data.csv")

print("Train:", train.shape)
print("Test:", test.shape)

train.head()

In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 5531451 entries, 0 to 5531450
Columns: 190 entries, customer_ID to D_145
dtypes: float64(185), int64(1), str(4)
memory usage: 8.2 GB


In [5]:
train["S_2"] = pd.to_datetime(train["S_2"])

history = train.groupby("customer_ID").agg(
    n_statements=("S_2", "size"),
    first_statement=("S_2", "min"),
    last_statement=("S_2", "max")
)

history["history_days"] = (
    history["last_statement"] - history["first_statement"]
).dt.days

history.describe()

,n_statements,first_statement,last_statement,history_days
count,458913.000000,458913,458913,458913.000000
mean,12.053376,2017-04-13 01:05:18.672166,2018-03-16 11:24:57.483836,337.430310
min,1.000000,2017-03-01 00:00:00,2018-03-01 00:00:00,0.000000
25%,13.000000,2017-03-11 00:00:00,2018-03-10 00:00:00,352.000000
50%,13.000000,2017-03-19 00:00:00,2018-03-17 00:00:00,365.000000
75%,13.000000,2017-03-28 00:00:00,2018-03-24 00:00:00,369.000000
max,13.000000,2018-03-31 00:00:00,2018-03-31 00:00:00,395.000000
std,2.610273,NaN,NaN,80.154861


In [6]:
history["n_statements"].value_counts().sort_index()

n_statements
1       5120
2       6098
3       5778
4       4673
5       4671
6       5515
7       5198
8       6110
9       6411
10      6721
11      5961
12     10623
13    386034
Name: count, dtype: int64

In [7]:
feature_groups = {
    prefix: [
        col for col in train.columns
        if col.startswith(prefix + "_")
    ]
    for prefix in ["D", "S", "P", "B", "R"]
}

for group, cols in feature_groups.items():
    print(group, len(cols))

D 96
S 22
P 3
B 40
R 28


In [8]:
for group, cols in feature_groups.items():
    missing = train[cols].isna().mean().mean()

    print(
        group,
        f"{missing:.2%}"
    )

D 23.44%
S 5.28%
P 2.09%
B 8.75%
R 6.63%


In [10]:
categorical_cols = [
    "B_30",
    "B_38",
    "D_114",
    "D_116",
    "D_117",
    "D_120",
    "D_126",
    "D_63",
    "D_64",
    "D_66",
    "D_68"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")

    print(
        train[col]
        .value_counts(dropna=False)
        .head(20)
    )


--- B_30 ---
B_30
0.0    4710663
1.0     763955
2.0      54817
NaN       2016
Name: count, dtype: int64

--- B_38 ---
B_38
2.0    1953232
3.0    1255315
1.0    1160047
5.0     444856
4.0     294917
7.0     259028
6.0     162040
NaN       2016
Name: count, dtype: int64

--- D_114 ---
D_114
1.0    3316478
0.0    2038257
NaN     176716
Name: count, dtype: int64

--- D_116 ---
D_116
0.0    5348109
NaN     176716
1.0       6626
Name: count, dtype: int64

--- D_117 ---
D_117
-1.0    1456084
 3.0    1166400
 4.0    1138666
 2.0     666808
 5.0     459290
 6.0     344520
 NaN     176716
 1.0     122967
Name: count, dtype: int64

--- D_120 ---
D_120
0.0    4729723
1.0     625012
NaN     176716
Name: count, dtype: int64

--- D_126 ---
D_126
 1.0    4262414
 0.0     891323
-1.0     260898
 NaN     116816
Name: count, dtype: int64

--- D_63 ---
D_63
CO    4119621
CR     930133
CL     438390
XZ      25786
XM      10556
XL       6965
Name: count, dtype: int64

--- D_64 ---
D_64
O      2913244
U    

In [11]:
for col in categorical_cols:
    print(f"\n--- {col} ---")

    print(
        train[col]
        .value_counts(
            normalize=True,
            dropna=False
        )
    )


--- B_30 ---
B_30
0.0    0.851614
1.0    0.138111
2.0    0.009910
NaN    0.000364
Name: proportion, dtype: float64

--- B_38 ---
B_38
2.0    0.353114
3.0    0.226941
1.0    0.209718
5.0    0.080423
4.0    0.053316
7.0    0.046828
6.0    0.029294
NaN    0.000364
Name: proportion, dtype: float64

--- D_114 ---
D_114
1.0    0.599567
0.0    0.368485
NaN    0.031947
Name: proportion, dtype: float64

--- D_116 ---
D_116
0.0    0.966855
NaN    0.031947
1.0    0.001198
Name: proportion, dtype: float64

--- D_117 ---
D_117
-1.0    0.263237
 3.0    0.210867
 4.0    0.205853
 2.0    0.120548
 5.0    0.083032
 6.0    0.062284
 NaN    0.031947
 1.0    0.022231
Name: proportion, dtype: float64

--- D_120 ---
D_120
0.0    0.855060
1.0    0.112992
NaN    0.031947
Name: proportion, dtype: float64

--- D_126 ---
D_126
 1.0    0.770578
 0.0    0.161137
-1.0    0.047166
 NaN    0.021119
Name: proportion, dtype: float64

--- D_63 ---
D_63
CO    0.744763
CR    0.168154
CL    0.079254
XZ    0.004662
XM    0

In [12]:
missing = (
    train
    .isna()
    .mean()
    .sort_values(ascending=False)
)

display(missing.head(30))

D_87     0.999301
D_88     0.998915
D_108    0.994768
D_110    0.994335
D_111    0.994335
B_39     0.993920
D_73     0.989902
B_42     0.987078
D_135    0.964801
D_134    0.964801
D_137    0.964801
D_138    0.964801
D_136    0.964801
R_9      0.943499
B_29     0.931046
D_106    0.902133
D_132    0.901911
D_49     0.901376
R_26     0.889847
D_76     0.887462
D_66     0.887307
D_42     0.856943
D_142    0.829266
D_53     0.738429
D_82     0.733734
D_50     0.568097
B_17     0.567229
D_105    0.546228
D_56     0.540716
S_9      0.530357
dtype: float64

Explore the labels

In [ ]:
labels = pd.read_csv("C:\\Users\\polok\\deep-learning-assignment-1\\data\\train_labels.csv")

target_counts = labels["target"].value_counts()
target_share = labels["target"].value_counts(normalize=True)

print(target_counts)
print(target_share)

target_counts.plot(kind="bar")
plt.xlabel("Target")
plt.ylabel("Customers")
plt.title("Target distribution")
plt.show()

Whether missingness itself has predictive signal.

In [ ]:
latest = (
    train
    .sort_values(["customer_ID", "S_2"])
    .groupby("customer_ID", sort=False)
    .tail(1)
    .merge(labels, on="customer_ID", how="left")
)

results = []

for col in train.columns:
    if col in ["customer_ID", "S_2"]:
        continue

    grouped = (
        latest
        .assign(is_missing=latest[col].isna())
        .groupby("is_missing")["target"]
        .agg(["mean", "count"])
    )

    if True in grouped.index and False in grouped.index:
        results.append({
            "feature": col,
            "default_rate_present": grouped.loc[False, "mean"],
            "default_rate_missing": grouped.loc[True, "mean"],
            "difference": grouped.loc[True, "mean"] - grouped.loc[False, "mean"]
        })

missing_target = pd.DataFrame(results)

display(
    missing_target
    .sort_values("difference", key=abs, ascending=False)
    .head(20)
)